# Modelvergelijking - DiffBrush vs DiffusionPen

Zet de OCR-readback resultaten van beide handwriting-generation modellen naast elkaar.

Leest `results/ocr_results_diffbrush.json` en `results/ocr_results_diffusionpen.json`. Run eerst beide eval-notebooks (`diffbrush_eval.ipynb` en `diffusionpen_eval.ipynb`) helemaal door zodat beide JSON-bestanden bestaan.

In [2]:
import json
from pathlib import Path

try:
    from IPython.display import display
except ImportError:
    display = print

# Vind de project root (map met results/), of de notebook nu geopend is
# vanuit notebooks/ of de project root.
for PROJECT_ROOT in [Path.cwd(), *Path.cwd().parents]:
    if (PROJECT_ROOT / 'results').is_dir():
        break
else:
    raise FileNotFoundError(
        'results/ map niet gevonden. Open de notebook vanuit de project root of de notebooks/ map.'
    )

db_path = PROJECT_ROOT / 'results' / 'ocr_results_diffbrush.json'
dp_path = PROJECT_ROOT / 'results' / 'ocr_results_diffusionpen.json'

if not db_path.exists() or not dp_path.exists():
    missing = [str(p.name) for p in (db_path, dp_path) if not p.exists()]
    print(f'Ontbrekend: {missing}. Run beide eval-notebooks helemaal door.')
else:
    db = json.loads(db_path.read_text(encoding='utf-8'))
    dp = json.loads(dp_path.read_text(encoding='utf-8'))

    # Samenvattings tabel
    summary = [
        {'Model': dp['model'], 'Gem. CER': dp['mean_CER'], 'Gem. WER': dp['mean_WER'],
         'Blokken': len(dp['rows'])},
        {'Model': db['model'], 'Gem. CER': db['mean_CER'], 'Gem. WER': db['mean_WER'],
         'Blokken': len(db['rows'])},
    ]

    # Per-adres tabel (gemiddelde CER over de writers)
    def per_addr(rows):
        agg = {}
        for r in rows:
            agg.setdefault(r['addr'], []).append(r['CER'])
        return {a: round(sum(v) / len(v), 3) for a, v in sorted(agg.items())}

    db_per = per_addr(db['rows'])
    dp_per = per_addr(dp['rows'])
    addrs = sorted(set(db_per) | set(dp_per))
    per_addr_rows = [
        {'Adres': a, 'DiffusionPen CER': dp_per.get(a, '-'), 'DiffBrush CER': db_per.get(a, '-')}
        for a in addrs
    ]

    try:
        import pandas as pd
        print('Samenvatting:')
        display(pd.DataFrame(summary))
        print('\nPer adres, gemiddelde CER over de writers:')
        display(pd.DataFrame(per_addr_rows))
    except ImportError:
        print('Samenvatting:', summary)
        print('Per adres :', per_addr_rows)

    # Verdict
    winner = 'DiffBrush' if db['mean_CER'] < dp['mean_CER'] else 'DiffusionPen'
    delta = abs(db['mean_CER'] - dp['mean_CER'])
    print(f'\nLaagste gemiddelde CER: {winner} (verschil {delta:.3f})')

Samenvatting:


,Model,Gem. CER,Gem. WER,Blokken
0,DiffusionPen,0.457,0.924,15
1,DiffBrush,0.416,0.730,15



Per adres, gemiddelde CER over de writers:


,Adres,DiffusionPen CER,DiffBrush CER
0,0,0.538,0.436
1,1,0.452,0.453
2,2,0.475,0.412
3,3,0.415,0.279
4,4,0.405,0.500



Laagste gemiddelde CER: DiffBrush (verschil 0.041)
